# Abnormal Trading Volume and 1–5 Day Return Reversal after Large Daily Price Moves in U.S. Common Stocks

**Research notebook**  
This notebook presents the research process in the order it actually developed: initial hypothesis → formal test → diagnostic failure → magnitude control → revised hypothesis → robustness → locked holdout.

It is designed to be **reviewer-readable**, not merely a dump of scripts. The heavy raw-data download and full historical sample-construction code remain in the repository's `.py` files. This notebook reproduces the main aggregate evidence from the derived result files and documents the methodological decisions that produced the final specification.

**Important data/security note:** raw Massive licensed data and API credentials are not embedded in this notebook or repository. The full acquisition scripts request the API key interactively with `getpass()` and were run only under the assessment's authorised network conditions.

## Notebook map

1. Research question and hypothesis evolution  
2. Literature motivation  
3. Data, universe and security  
4. Variable definitions and parameter rationale  
5. Initial formal result and failed interpretation  
6. Move-magnitude confound diagnostic  
7. 1%-rank magnitude control  
8. Main development results  
9. Continuous residual-magnitude control  
10. Parameter robustness  
11. Locked holdout  
12. Interpretation, limitations and future work  
13. Experiment record, decision log and source/tool disclosure

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display, Image, Markdown

ROOT = Path('.')
RESULTS = ROOT / 'results'
FINAL = RESULTS / 'final_outputs'
DOCS = ROOT / 'documentation'

required = [
    RESULTS / 'formal_development_daily_spreads.csv',
    RESULTS / 'move_magnitude_diagnostic.csv',
    RESULTS / 'magnitude_control_balance.csv',
    RESULTS / 'magnitude_controlled_development.csv',
    RESULTS / 'development_robustness_daily.csv',
    RESULTS / 'locked_holdout_daily.csv',
    FINAL / 'figure2_positive_negative_asymmetry.png',
    FINAL / 'figure3_negative_5d_robustness.png',
    FINAL / 'figure1_negative_development_vs_holdout.png',
]

missing = [str(p) for p in required if not p.exists()]
print('Notebook root:', ROOT.resolve())
print('Required derived files present:', len(missing) == 0)
if missing:
    print('Missing:', missing)

# 1. Research Question and Hypothesis Evolution

### Final research question

> Among point-in-time U.S. common stocks priced at least $5 with trailing 20-day average dollar volume of at least $1 million, does abnormal trading volume on a large-move day—measured as current trading volume relative to the stock's previous 20-trading-day average—contain incremental information about cumulative returns over the following 1, 3 and 5 trading days, after conditioning on the direction and magnitude of the signal-day close-to-close return?

### Initial hypothesis

The initial hypothesis predicted **same-direction continuation**: after a large daily move, unusually high abnormal volume would be associated with stronger continuation over the next 1D, 3D and 5D.

### Revised hypothesis

Development evidence contradicted the negative-side continuation prediction. After diagnosing and controlling for signal-day move magnitude, the revised hypothesis became:

> Following large negative daily price moves, unusually high abnormal trading volume is associated with stronger return reversal over the following 3–5 trading days, while the corresponding relationship following large positive moves is substantially weaker.

The revised hypothesis and performance-related main methodology were frozen **before holdout return performance was examined**.

# 2. Literature Motivation

Three papers motivated different components of the question rather than serving as exact replications:

- **Jegadeesh & Titman (1993)** motivates the broader idea that past returns may contain information about subsequent returns, although their classic momentum setting is much longer-horizon than this project.
- **Lee & Swaminathan (2000)** motivates treating trading activity as an additional state variable beyond price history alone.
- **Gökçen & Post (2018)** is closer to the short-horizon setting because it studies trading volume, return variability and short-term momentum.

This project differs by conditioning specifically on large daily price moves, using stock-specific abnormal volume, separating positive and negative movers, testing 1D/3D/5D outcomes, and explicitly controlling the magnitude of the signal-day move before comparing High- and Low-AVOL stocks.

# 3. Data, Universe Construction and Security

The study uses Massive U.S. equity data from 2015–2025:

- **Daily Market Summary / Aggregate Bars** for adjusted daily OHLCV;
- **Historical Ticker Reference** for point-in-time U.S. common-stock (`type=CS`) universes.

The historical common-stock universe is refreshed on the **first valid trading day of each month**. Daily observations are intersected with that month's historical universe before the price and liquidity filters are applied.

### Primary universe filters

- Signal-day price ≥ **$5**
- Trailing **ADV20 ≥ $1M**, where ADV20 is the average of `Close × Volume` over the previous 20 consecutive valid market trading days
- Sufficient consecutive history to calculate the previous close, AVOL20 and ADV20

The price filter reduces very-low-priced-stock and microstructure concerns. The liquidity threshold was chosen before formal factor-performance evaluation using the monthly-snapshot liquidity distribution, and $5M/$10M thresholds are reserved for robustness checks.

### Credential handling

The acquisition scripts use interactive `getpass()` input. No API key is written to notebook cells, source code, `.env` files, outputs, or Git history. Raw licensed Massive data are intentionally excluded from the submitted repository.

In [ ]:
# The following is a NON-NETWORK security illustration only.
# The real acquisition scripts are download_full_daily.py and download_monthly_universes.py.
# No API call is made by this cell.

safe_acquisition_pattern = '''
from getpass import getpass
API_KEY = getpass("Enter Massive API key: ")
headers = {"Authorization": f"Bearer {API_KEY}"}
# Network location was checked before Massive access in the acquisition scripts.
'''
print(safe_acquisition_pattern)

## 3.1 Liquidity-threshold diagnostic

The $1M ADV20 threshold was not selected by trying multiple cutoffs and choosing the best backtest. Before formal factor-performance evaluation, I inspected the ADV20 distribution at monthly point-in-time universe snapshot dates among common stocks already satisfying the $5 price filter.

In [ ]:
liq_text = (RESULTS / 'liquidity_distribution.txt').read_text()
start = liq_text.index('COMMON STOCKS WITH PRICE >= $5')
print(liq_text[start:])

**Decision.** The 25th percentile was about $1.11M and about 76% of price-filtered observations exceeded $1M. I therefore used **ADV20 ≥ $1M** as the main broad-but-investable liquidity screen, while $5M and $10M were treated only as robustness checks.

# 4. Variable Definitions and Parameter Rationale

### Signal-day return

$$
r_{i,t} = \frac{Close_{i,t}}{Close_{i,t-1}} - 1
$$

This is **close-to-close**, so it captures both overnight and intraday adjustment. A future extension can decompose overnight and open-to-close components.

### Abnormal volume

$$
AVOL20_{i,t}=\frac{Volume_{i,t}}{\frac{1}{20}\sum_{j=1}^{20} Volume_{i,t-j}}
$$

The signal day is excluded from the trailing average. A value of 2 means current volume is twice the prior-20-day average.

### Future outcomes

$$
R^{(h)}_{i,t}=\frac{Close_{i,t+h}}{Close_{i,t}}-1,\quad h\in\{1,3,5\}
$$

### Main parameters and rationale

| Component | Main choice | Rationale |
|---|---|---|
| Mover tail | Top / bottom 20% | Large relative moves while retaining enough stocks for narrow within-tail matching |
| AVOL lookback | 20D | Roughly one trading month; balances recency and stability |
| Future horizons | 1D / 3D / 5D | Immediate, multi-day and approximately one-week adjustment |
| Price screen | $5 | Reduces influence of very low-priced stocks |
| Liquidity | ADV20 ≥ $1M | Removes least-liquid portion without collapsing breadth |
| Magnitude control | 1%-rank bins | Chosen after 5%-bins still left material signal-day imbalance |
| AVOL split | Within-bin median | Balanced relative High/Low groups without an arbitrary absolute AVOL cutoff |
| Inference | HAC lag = h−1 | Accounts for serial correlation from overlapping h-day outcomes |

In [ ]:
# Main locked parameters, matching the source implementation.
PRICE_MIN = 5.0
ADV20_MIN = 1_000_000
AVOL_LOOKBACK = 20
MOVER_TAIL = 0.20
HORIZONS = [1, 3, 5]
HAC_LAGS = {h: h - 1 for h in HORIZONS}

pd.Series({
    'Price minimum': PRICE_MIN,
    'ADV20 minimum': ADV20_MIN,
    'AVOL lookback': AVOL_LOOKBACK,
    'Mover tail': MOVER_TAIL,
    'Horizons': HORIZONS,
    'HAC lags': HAC_LAGS,
})

## 4.1 Core magnitude-control implementation

On each day, eligible stocks are ranked cross-sectionally by signal-day return. The top 20% and bottom 20% tails are each divided into **twenty 1-percentile return-rank bins**. Within each bin, AVOL is split at the bin median.

The primary implementation requires all 20 bins to contain valid High- and Low-AVOL groups. At a given future horizon, the signal date is retained only if all 20 bins continue to have valid future-return observations on both sides. The daily spread is the equal-weight mean of the 20 bin-level High-minus-Low spreads.

In [ ]:
# Compact representation of the exact core sorting logic used by the project.

def build_groups_for_review(day_df, mover):
    if mover == 'positive':
        group = day_df[day_df['return_rank'] >= 0.80].copy()
        edges = np.round(np.arange(0.80, 1.001, 0.01), 2)
    else:
        group = day_df[day_df['return_rank'] <= 0.20].copy()
        edges = np.round(np.arange(0.00, 0.201, 0.01), 2)

    labels = [f'{edges[i]:.2f}-{edges[i+1]:.2f}' for i in range(len(edges)-1)]
    group['magnitude_bin'] = pd.cut(
        group['return_rank'], bins=edges, labels=labels,
        include_lowest=True, right=True
    )

    out = {}
    for label in labels:
        b = group[group['magnitude_bin'] == label]
        if b.empty:
            continue
        med = b['avol'].median()
        low = b[b['avol'] < med]
        high = b[b['avol'] >= med]
        if low.empty or high.empty:
            continue
        out[label] = {'low': low, 'high': high}

    return out if len(out) == 20 else None

print('Expected number of 1%-rank bins per mover tail: 20')

# 5. Initial Formal Result: A Result That Could Not Yet Be Interpreted

The first formal specification used the point-in-time universe, price/liquidity filters, top/bottom-20% mover tails and one broad High/Low AVOL split per tail. It produced positive High-minus-Low spreads for **negative movers**, opposite to the original continuation prediction.

Rather than treating this immediately as a reversal finding, I investigated whether High-AVOL stocks also had more extreme signal-day moves.

In [ ]:
def hac_mean(series, horizon):
    x = pd.Series(series).dropna().astype(float)
    X = np.ones((len(x), 1))
    fit = sm.OLS(x.values, X).fit(
        cov_type='HAC', cov_kwds={'maxlags': horizon - 1}
    )
    return pd.Series({
        'N': len(x),
        'Spread (bps)': fit.params[0] * 10000,
        'HAC SE (bps)': fit.bse[0] * 10000,
        'HAC t-stat': fit.tvalues[0],
    })

coarse = pd.read_csv(RESULTS / 'formal_development_daily_spreads.csv')
rows = []
for mover in ['positive', 'negative']:
    for h in HORIZONS:
        x = coarse[(coarse.mover == mover) & (coarse.horizon == h)]
        s = hac_mean(x['spread'], h)
        s['Mover'] = mover.capitalize()
        s['Horizon'] = f'{h}D'
        rows.append(s)
coarse_summary = pd.DataFrame(rows)[['Mover','Horizon','N','Spread (bps)','HAC t-stat']]
display(coarse_summary.round(2))

**Interpretation.** A positive spread for negative movers means High-AVOL losers subsequently outperform Low-AVOL losers—i.e., **reversal**, not continuation. But at this stage the AVOL groups were not adequately matched on the initial move magnitude, so the result was not yet interpreted as incremental AVOL information.

# 6. Diagnostic: Signal-Day Move-Magnitude Confound

The key diagnostic asks whether High-AVOL and Low-AVOL stocks inside the broad mover tails experienced comparable initial price moves.

In [ ]:
diag = pd.read_csv(RESULTS / 'move_magnitude_diagnostic.csv')
diag_summary = diag.groupby('mover').agg(
    Low_signal_return=('low_signal_return','mean'),
    High_signal_return=('high_signal_return','mean'),
    High_minus_Low=('high_minus_low_signal_return','mean'),
    Low_mean_rank=('low_mean_rank','mean'),
    High_mean_rank=('high_mean_rank','mean'),
)
for c in ['Low_signal_return','High_signal_return','High_minus_Low']:
    diag_summary[c] *= 100

display(diag_summary.rename(columns={
    'Low_signal_return':'Low signal return (%)',
    'High_signal_return':'High signal return (%)',
    'High_minus_Low':'High-Low signal gap (pp)',
}).round(3))

**Diagnostic conclusion.** High-AVOL positive movers rose more on the signal day, and High-AVOL negative movers fell more. For negative movers, the broad-sort averages were roughly **−3.65% vs −2.84%**. Therefore a stronger subsequent rebound could simply reflect a larger initial decline.

### Failed 5%-bin control

I first divided each 20% tail into four 5-percentage-point return-rank bins and re-split AVOL within each bin. This improved balance, but the remaining High-minus-Low signal-day gaps were still approximately **+68.2 bps** for positive movers and **−38.7 bps** for negative movers.

**Decision:** refine to 1%-rank bins based on **balance**, before using the controlled future-return performance to choose the specification.

# 7. Primary 1%-Rank Magnitude Control

The final main design divides each 20% mover tail into twenty 1-percentile rank bins, then performs the AVOL median split inside each bin. This was intended to compare stocks with very similar initial return ranks rather than simply comparing all High-AVOL and Low-AVOL movers together.

In [ ]:
bal = pd.read_csv(RESULTS / 'magnitude_control_balance.csv')
balance_summary = bal.groupby('mover').agg(
    Signal_days=('date','nunique'),
    Low_signal_return=('low_signal_return','mean'),
    High_signal_return=('high_signal_return','mean'),
    Signal_gap=('signal_return_difference','mean'),
    Low_rank=('low_rank','mean'),
    High_rank=('high_rank','mean'),
)
for c in ['Low_signal_return','High_signal_return','Signal_gap']:
    balance_summary[c] *= 10000

display(balance_summary.rename(columns={
    'Low_signal_return':'Low signal return (bps)',
    'High_signal_return':'High signal return (bps)',
    'Signal_gap':'High-Low signal gap (bps)',
}).round(2))

**Balance result.** Mean cross-sectional return ranks became almost identical between High- and Low-AVOL groups. The remaining signal-day return gap fell to about **+28.1 bps** for positive movers and **−15.8 bps** for negative movers. I stopped refining the bin width here rather than continuing until a preferred future-return result appeared.

# 8. Main Development Results

The primary development sample uses actual signal dates from **2015-02-02 through 2025-01-24**. Late-January signals are excluded so that the longest 5D development outcomes finish before the February 2025 holdout begins.

In [ ]:
dev = pd.read_csv(RESULTS / 'magnitude_controlled_development.csv', parse_dates=['signal_date'])

main_rows = []
for mover in ['positive','negative']:
    for h in HORIZONS:
        x = dev[(dev.mover == mover) & (dev.horizon == h)].copy()
        hs = hac_mean(x['spread'], h)
        hs['Mover'] = mover.capitalize()
        hs['Horizon'] = f'{h}D'
        hs['Low AVOL future (bps)'] = x['low_avol_return'].mean() * 10000
        hs['High AVOL future (bps)'] = x['high_avol_return'].mean() * 10000
        hs['Signal gap (bps)'] = (x['high_signal_return'] - x['low_signal_return']).mean() * 10000
        main_rows.append(hs)

main_dev = pd.DataFrame(main_rows)[[
    'Mover','Horizon','N','Low AVOL future (bps)','High AVOL future (bps)',
    'Spread (bps)','HAC t-stat','Signal gap (bps)'
]]
display(main_dev.round(2))

### Main interpretation

- **Positive movers:** spreads are small and HAC t-statistics are weak.
- **Negative movers:** High-minus-Low spreads are approximately **+4.52, +12.89 and +21.04 bps** at 1D/3D/5D, with HAC t-statistics approximately **4.10, 5.29 and 5.50**.
- Because these stocks had just experienced large **negative** returns, a positive future High-minus-Low spread means the High-AVOL group subsequently rebounds more strongly: **reversal rather than continuation**.
- The increasing spread from 1D to 5D suggests that the relationship is not only an immediate next-day bounce.

In [ ]:
display(Image(filename=str(FINAL / 'figure2_positive_negative_asymmetry.png')))

**Figure 1 interpretation.** The positive-mover confidence intervals overlap zero, while the negative-mover High-minus-Low spreads are larger and increase from 1D to 5D. This asymmetry is the empirical reason the hypothesis was revised toward negative-mover reversal rather than symmetric continuation.

# 9. Continuous Control for the Residual Signal-Day Gap

The 1%-bin design greatly improves rank balance but does not make the average signal-day returns exactly equal. I therefore estimate, for each mover and horizon:

$$
Spread_{t,h}=\alpha_h+\beta_h\,SignalGap_t+\epsilon_{t,h}
$$

where `SignalGap = High-AVOL signal-day return − Low-AVOL signal-day return`. The intercept \(\alpha_h\) estimates the High-minus-Low future spread at a zero signal-day gap, conditional on this linear control. HAC standard errors again use lag `h−1`.

In [ ]:
def continuous_control(df, horizon):
    y = df['spread'].astype(float)
    gap = (df['high_signal_return'] - df['low_signal_return']).astype(float)
    X = sm.add_constant(gap)
    fit = sm.OLS(y, X).fit(
        cov_type='HAC', cov_kwds={'maxlags': horizon - 1}
    )
    return pd.Series({
        'N': len(df),
        'Raw spread (bps)': y.mean() * 10000,
        'Mean signal gap (bps)': gap.mean() * 10000,
        'Adjusted spread (bps)': fit.params['const'] * 10000,
        'Adjusted HAC t-stat': fit.tvalues['const'],
        'R-squared': fit.rsquared,
    })

control_rows = []
for mover in ['positive','negative']:
    for h in HORIZONS:
        x = dev[(dev.mover == mover) & (dev.horizon == h)].copy()
        s = continuous_control(x, h)
        s['Mover'] = mover.capitalize(); s['Horizon'] = f'{h}D'
        control_rows.append(s)
control = pd.DataFrame(control_rows)[[
    'Mover','Horizon','N','Raw spread (bps)','Mean signal gap (bps)',
    'Adjusted spread (bps)','Adjusted HAC t-stat','R-squared'
]]
display(control.round(3))

**Interpretation.** The adjustment reduces the negative-mover effect, especially at 1D, showing that residual initial-move differences explain part of the raw spread. However, the negative 3D and 5D adjusted spreads remain positive at approximately **+8.32 bps** and **+16.85 bps**. This is why the revised pre-holdout hypothesis emphasized the **3–5 day** reversal.

# 10. Parameter Robustness

The purpose of robustness analysis is not to search for the best-looking parameter combination. The main specification is retained while one dimension is varied at a time:

- AVOL lookback: 10D / 20D / 40D
- ADV20 threshold: $1M / $5M / $10M
- mover tail: 20% / 10%

The robustness runner additionally requires at least four stocks per 1%-rank bin before the within-bin AVOL split. This produces a slightly different baseline sample (**2,503 dates**) from the primary development sample (**2,506 dates**), explaining the very small difference between the main primary and robustness-baseline numbers.

In [ ]:
rob = pd.read_csv(FINAL / 'table2_negative_robustness.csv')
display(rob.round(2))

In [ ]:
display(Image(filename=str(FINAL / 'figure3_negative_5d_robustness.png')))

**Figure 2 interpretation.** The negative-mover 5D spread remains positive across shorter/longer AVOL baselines, stricter liquidity thresholds and a more extreme 10% mover tail. The 10% tail produces the largest estimate, but it is deliberately **not** adopted as the main model because selecting the best-performing robustness variant after seeing results would introduce specification-selection bias.

# 11. Locked Holdout Evaluation

The revised hypothesis and main performance-related parameters were frozen before holdout return performance was examined. The actual holdout contains **225 valid signal dates from 2025-02-03 through 2025-12-23**; the end date leaves enough subsequent trading days to calculate the 5D outcome before the end of the data.

A liquidity-distribution diagnostic did use observations extending through 2025 before formal return-performance evaluation, so the precise claim is **not** that every holdout datum was completely untouched. The claim is that **holdout return performance was not examined before the revised hypothesis and performance methodology were locked**.

In [ ]:
hold = pd.read_csv(RESULTS / 'locked_holdout_daily.csv', parse_dates=['signal_date'])

hold_rows = []
for mover in ['positive','negative']:
    for h in HORIZONS:
        x = hold[(hold.mover == mover) & (hold.horizon == h)].copy()
        hs = hac_mean(x['spread'], h)
        adj = continuous_control(x, h)
        hold_rows.append({
            'Mover': mover.capitalize(),
            'Horizon': f'{h}D',
            'N': len(x),
            'Spread (bps)': hs['Spread (bps)'],
            'HAC t-stat': hs['HAC t-stat'],
            'Adjusted spread (bps)': adj['Adjusted spread (bps)'],
            'Adjusted HAC t-stat': adj['Adjusted HAC t-stat'],
        })
hold_summary = pd.DataFrame(hold_rows)
display(hold_summary.round(2))

In [ ]:
display(Image(filename=str(FINAL / 'figure1_negative_development_vs_holdout.png')))

**Figure 3 interpretation.** The negative-mover High-minus-Low spread remains positive at 1D, 3D and 5D in the holdout, so the **direction** of the revised relationship replicates outside the development sample. The holdout magnitude is much larger, however, so the holdout is better interpreted as evidence for directional replication than for a stable economic effect size. Regime dependence is a plausible explanation and is left for future research.

# 12. Combined Interpretation

The evidence supports three main conclusions:

1. The original **symmetric continuation** hypothesis is not supported.
2. The strongest relationship is among **negative movers**, where High-AVOL stocks subsequently outperform Low-AVOL stocks—consistent with **reversal**.
3. The negative-side relationship is strongest over **3D–5D**, remains positive after continuous residual-magnitude control, survives nearby parameter variations, and replicates in direction in the locked holdout.

The result is a **predictive association**, not a causal claim. The High-minus-Low spreads are also not equivalent to directly implementable PnL because the signal uses the completed signal-day close and volume, while execution costs and next-tradable-price effects are not modelled.

## 12.1 Limitations and future improvements

The main limitations are deliberately not “fixed” after observing the holdout, because repeatedly adding filters after seeing final performance would weaken the out-of-sample interpretation.

- **Causality / omitted events:** AVOL may coincide with earnings, news, analyst revisions, attention or liquidity shocks. Future work should separate event and non-event days.
- **Daily aggregation:** close-to-close returns combine overnight and intraday adjustment. Future work could decompose overnight returns, open-to-close returns and intraday volume timing.
- **Recent listings:** the consecutive 20-day history requirement excludes the newest IPO observations, but there is no additional seasoning-period rule. A minimum listing-age robustness test is a natural extension.
- **Monthly universe refresh:** substantially better than a current-universe backtest, but not a fully daily point-in-time universe.
- **Liquidity / microstructure:** Price ≥ $5 and ADV20 ≥ $1M do not fully control bid-ask spreads, trade count, depth, halts or price impact.
- **Other stock characteristics:** matching is tight on signal-day return rank, but not on size, volatility, beta, industry or longer-horizon returns.
- **Execution realism:** the signal requires full day-t close and volume; a real implementation would need an explicit post-signal execution time and price.
- **Costs:** commissions, spreads, slippage, impact, borrowing costs and short-sale constraints are not modelled.
- **Regime dependence:** holdout estimates are much larger than development estimates.
- **Multiple testing:** several horizons and robustness variants were inspected; future work could preregister the full empirical design or use formal multiple-testing adjustments.

# 13. Experiment Record

The project deliberately records failed and superseded experiments rather than presenting only the final specification. The complete table is maintained in `documentation/experiment_record.md`.

In [ ]:
# Render the maintained experiment record inside the notebook.
display(Markdown((DOCS / 'experiment_record.md').read_text()))

# 14. Research Decision Log

The decision log records the observations that changed the research design, including the prototype sampling bug, universe construction, the move-magnitude confound, the failed 5%-bin control, the switch to 1%-bins based on balance, the hypothesis revision and the pre-holdout freeze.

In [ ]:
display(Markdown((DOCS / 'decision_log.md').read_text()))

# 15. Sources and AI Tool Disclosure

### Academic sources

- Jegadeesh, N. & Titman, S. (1993), *Returns to Buying Winners and Selling Losers: Implications for Stock Market Efficiency*, The Journal of Finance.
- Lee, C. M. C. & Swaminathan, B. (2000), *Price Momentum and Trading Volume*, The Journal of Finance.
- Gökçen, U. & Post, T. (2018), *Trading Volume, Return Variability and Short-Term Momentum*, The European Journal of Finance.

### Data

Massive U.S. equity Daily Market Summary / Aggregate Bars and Historical Ticker Reference data. No external market dataset is used in the final analysis. Raw Massive licensed data are not included in the submitted repository.

### AI assistance

ChatGPT (OpenAI) was used for conceptual clarification, research-design discussion, Python coding/debugging assistance, statistical-method explanation, and organisation/editing of the research documentation. All code was run locally, outputs were inspected by the researcher, and the researcher remains responsible for understanding and defending the methodology, code and conclusions. The Massive API credential was never shared with the AI.

### External code

No external research repository or third-party trading-strategy code was copied into the final analysis. Standard Python package documentation was consulted where necessary.

# 16. Reproducibility Notes

The notebook intentionally operates on **derived aggregate research outputs** already produced by the full pipeline. This keeps the reviewer-facing notebook readable and avoids redistributing raw licensed Massive data.

The complete raw-data and sample-construction workflow remains available in the repository scripts, including:

- `download_full_daily.py`
- `download_monthly_universes.py`
- `inspect_liquidity_distribution.py`
- `build_formal_development.py`
- `diagnose_move_magnitude.py`
- `build_magnitude_controlled_development.py`
- `evaluate_magnitude_controlled_development.py`
- `continuous_return_control.py`
- `run_development_robustness.py`
- `run_locked_holdout.py`
- `generate_final_outputs.py`

The notebook should be read together with the Final Report, README, experiment record, decision log and source/tool disclosure. The Final Report is the polished research narrative; this notebook is the code-and-evidence trail supporting it.